- Extracting U-I interactions and performing 5-core, re-indexing

In [1]:
import os
import pandas as pd

In [ ]:

dataset_name='MicroLens-100k'

folder = './datasets/%s/raw'%dataset_name
os.chdir(folder) 
os.getcwd()
k_core = 5
if not os.path.exists('../%s-core'%k_core): 
    os.makedirs('../%s-core'%k_core)


## 5-core filtering

In [3]:
df = pd.read_csv(f'{dataset_name}_pairs.csv', dtype={'user': str, 'item': str, 'timestamp': int})
df = df.rename(columns={'user': 'userID', 'item': 'itemID'})

print(f'shape: {df.shape}')
df[:5]

shape: (719405, 3)


,userID,itemID,timestamp
0,36121,9580,1583378629552
1,26572,9580,1583436719018
2,94805,9580,1584083806481
3,37550,9580,1584412681021
4,89825,9580,1584649439020


In [ ]:

learner_id, course_id, tmstmp_str = 'userID', 'itemID', 'timestamp'
df.dropna(subset=[learner_id, course_id, tmstmp_str], inplace=True) 
print(df.shape)
df.drop_duplicates(subset=[learner_id, course_id, tmstmp_str], inplace=True) # 
print(f'After dropped: {df.shape}')
print('Original #inter:', len(df))
print('Original #item:', len(set(df['itemID'])))
print('Original #user:', len(set(df['userID'])))
df[:3]

(719405, 3)
After dropped: (719405, 3)
Original #inter: 719405
Original #item: 19738
Original #user: 100000


,userID,itemID,timestamp
0,36121,9580,1583378629552
1,26572,9580,1583436719018
2,94805,9580,1584083806481


## k-core

In [5]:
def filter_by_k_core(df, k):
    del_item_num, del_user_num = 100, 100
    while del_item_num > 0 or del_user_num > 0:
        ## drop items
        item_count = pd.DataFrame(df['itemID'].value_counts())
        item_count.columns = ['nums']
        item_count = item_count[item_count['nums'] < k]
        item_delindexs = item_count.index
        del_item_num = len(item_delindexs)
        ## drop users
        user_count = pd.DataFrame(df['userID'].value_counts())
        user_count.columns = ['nums']
        user_count = user_count[user_count['nums'] < k]
        user_delindexs = user_count.index
        del_user_num = len(user_delindexs)
        if del_item_num == 0 and del_user_num == 0:
            print("Finish droping, keeping core users-{} and items-{}.".format(k, k))
            break
        df = df[~df['itemID'].isin(item_delindexs)]
        df = df[~df['userID'].isin(user_delindexs)]

    return df

In [6]:
df = filter_by_k_core(df, k_core)  
print(f'shape after k-core: {df.shape}')
print('after k-core #inter:', len(df))
print('after k-core #item:', len(set(df['itemID'])))
print('after k-core #user:', len(set(df['userID'])))
df[:5]

Finish droping, keeping core users-5 and items-5.
shape after k-core: (705174, 3)
after k-core #inter: 705174
after k-core #item: 17228
after k-core #user: 98129


,userID,itemID,timestamp
0,36121,9580,1583378629552
1,26572,9580,1583436719018
2,94805,9580,1584083806481
3,37550,9580,1584412681021
4,89825,9580,1584649439020


## Re-index

In [7]:
i_mapping_file = 'i_id_mapping.csv'
u_mapping_file = 'u_id_mapping.csv'


uid_field, iid_field = learner_id, course_id

uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

# start from 1
u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

# dump
rslt_dir = '../%s-core'%k_core+'/'
 
u_df = pd.DataFrame(list(u_id_map.items()), columns=['user_id', 'userID'])
i_df = pd.DataFrame(list(i_id_map.items()), columns=['asin', 'itemID'])

u_df.to_csv(os.path.join(rslt_dir, u_mapping_file), sep='\t', index=False)
i_df.to_csv(os.path.join(rslt_dir, i_mapping_file), sep='\t', index=False)
print(f'mapping dumped...')

mapping dumped...


# get user/item interaction history

In [8]:
userid_his = list(df.sort_values(['userID','timestamp'],ascending=True).groupby('userID'))
user_hist_dict = {}
for user_record in userid_his:
    userid = user_record[0]
    record = user_record[1]
    item_list = list(record['itemID'])
    t_lit = list(record['timestamp'])
    user_hist_dict[userid] = [i for i in item_list]

dict(list(user_hist_dict.items())[:2])

{0: [0, 4746, 8903, 12911, 14072, 15478, 11501], 1: [0, 995, 5415, 2163, 8320]}

# data splitting
- Based on generated interactions, perform data splitting

In [9]:
print('Split dataset: ')
print(' Dataset: ', dataset_name)

# generate train valid test
train_inters, valid_inters, test_inters = dict(), dict(), dict()
for u_index in range(len(u_id_map)):
    inters = user_hist_dict[u_index]
    # leave one out
    train_inters[u_index] = [str(i_index) for i_index in inters[:-2]]
    valid_inters[u_index] = [str(inters[-2])]
    test_inters[u_index] = [str(inters[-1])]
    assert len(user_hist_dict[u_index]) == len(train_inters[u_index]) + \
            len(valid_inters[u_index]) + len(test_inters[u_index])

Split dataset: 
 Dataset:  MicroLens-100k


In [10]:
# save
print('Convert dataset: ')
print(' Dataset: ', dataset_name)
uid_list = list(train_inters.keys())
uid_list.sort(key=lambda t: int(t))

with open(os.path.join('../', f'{dataset_name}.train.inter'), 'w') as file:
    file.write('user_id:token\titem_id_list:token_seq\titem_id:token\n')
    for uid in uid_list:
        item_seq = train_inters[uid]
        seq_len = len(item_seq)
        for target_idx in range(1, seq_len):
            target_item = item_seq[-target_idx]
            seq = item_seq[:-target_idx][-50:]
            file.write(f'{uid}\t{" ".join(seq)}\t{target_item}\n')

with open(os.path.join('../', f'{dataset_name}.valid.inter'), 'w') as file:
    file.write('user_id:token\titem_id_list:token_seq\titem_id:token\n')
    for uid in uid_list:
        item_seq = train_inters[uid][-50:]
        target_item = valid_inters[uid][0]
        file.write(f'{uid}\t{" ".join(item_seq)}\t{target_item}\n')

with open(os.path.join('../', f'{dataset_name}.test.inter'), 'w') as file:
    file.write('user_id:token\titem_id_list:token_seq\titem_id:token\n')
    for uid in uid_list:
        item_seq = (train_inters[uid] + valid_inters[uid])[-50:]
        target_item = test_inters[uid][0]
        file.write(f'{uid}\t{" ".join(item_seq)}\t{target_item}\n')

Convert dataset: 
 Dataset:  MicroLens-100k
